# M1-B Qwen3-1.7B GPU Smoke Test

This notebook validates the GPU workflow required by Linear issue M1-04. It is a feasibility test, not a benchmark result and not full Model 1 training.

The notebook performs the following checks:

1. Captures the NVIDIA, CUDA, PyTorch, Transformers, PEFT and bitsandbytes environment.
2. Resolves and records the exact Hugging Face model revision used.
3. Loads M1 candidate `m1-b` (`Qwen/Qwen3-1.7B`) in 4-bit NF4 mode.
4. Adds LoRA adapters and runs a short backward training smoke test.
5. Saves a Trainer checkpoint and resumes from it.
6. Generates one structured M1 catch-up response.
7. Tests a 4096-token forward pass separately from the short training run.
8. Writes a machine-readable run manifest.

Run this notebook from a clean clone of the repository on an NVIDIA GeForce RTX 5080 Laptop with 16 GB. Model repo, revision, and LoRA settings come from `eval/configs/m1-b.yaml`. Large model files and checkpoints remain outside Git under the configured artifact directory.

## Before running

- Reserve a university GPU and confirm that `nvidia-smi` works.
- Use a fresh Python 3.10+ environment with a CUDA-compatible PyTorch 2.6+ installation supplied by the lab. The notebook deliberately does not replace PyTorch.
- Ensure at least 25 GB of free disk space for the model cache and checkpoints.
- If Hugging Face requires authentication, run `huggingface-cli login` in the terminal or set `HF_TOKEN` outside the notebook. Never paste a token into a cell.
- If the home-directory quota is small, set both `HF_HOME` and `M1_SMOKE_ARTIFACT_ROOT` to a lab scratch path before starting Jupyter.
- Restart the kernel after the package-installation cell if Jupyter requests it.
- Do not run `pip install -r requirements-models.txt` afterward: its older Transformers pin cannot load Qwen3. The notebook installs an isolated smoke-test stack.

This notebook intentionally validates only M1 candidate `m1-b`, the smallest generative candidate and the fastest way to validate the shared QLoRA/checkpoint workflow. M1-C (Qwen3.5-4B), M1-D (Gemma 4 E4B), and the extractive M1-A candidate require model-specific loader checks before their full runs.

In [ ]:
# Run once in a fresh GPU-lab environment. PyTorch is intentionally omitted so
# the notebook uses the CUDA-compatible build maintained by the lab.
%pip install -q -U "transformers>=4.51.0,<5" "accelerate>=1.0.0,<2" "peft>=0.14.0,<1" "bitsandbytes>=0.45.0,<1" "huggingface_hub>=0.27.0,<2" "safetensors>=0.4.5,<1" "pyyaml>=6.0,<7" "packaging>=24,<27"
print()
print("=" * 70)
print("  STEP 0 — DEPENDENCIES INSTALLED")
print("  Restart kernel if prompted by Jupyter.")
print("=" * 70)

In [ ]:
import json
import math
import os
import platform
import random
import subprocess
import sys
import time
from datetime import datetime, timezone
from importlib.metadata import version
from pathlib import Path

import torch
import yaml
from packaging.version import Version
from torch.utils.data import Dataset

assert sys.version_info >= (3, 10), f"Python 3.10+ is required; found {platform.python_version()}."
torch_version = Version(torch.__version__.split("+", 1)[0])
assert torch_version >= Version("2.6.0"), (
    f"PyTorch 2.6+ is required for safe Trainer checkpoint resume; found {torch.__version__}. "
    "Select a newer lab CUDA environment, then restart the kernel."
)
assert torch.cuda.is_available(), "CUDA is not available. Stop and select a GPU-backed Jupyter kernel."

def command_output(command):
    try:
        return subprocess.check_output(command, text=True, stderr=subprocess.STDOUT).strip()
    except Exception as exc:
        return f"unavailable: {exc}"

gpu = torch.cuda.get_device_properties(0)
gpu_total_gib = gpu.total_memory / 1024**3
assert gpu_total_gib >= 15, f"Expected the lab RTX 5080 Laptop (16 GB); detected only {gpu_total_gib:.1f} GiB."
compute_capability = torch.cuda.get_device_capability(0)
gpu_free_bytes, _ = torch.cuda.mem_get_info(0)
gpu_free_gib = gpu_free_bytes / 1024**3
assert gpu_free_gib >= 12, (
    f"At least 12 GiB free VRAM is required for this smoke test; found {gpu_free_gib:.1f} GiB. "
    "Release other GPU processes or reserve a different GPU."
)
try:
    cuda_probe = torch.randn((128, 128), device="cuda", dtype=torch.float16)
    cuda_probe_checksum = float((cuda_probe @ cuda_probe).abs().mean().item())
    torch.cuda.synchronize()
    del cuda_probe
except Exception as exc:
    raise RuntimeError(
        "CUDA was detected, but a test kernel failed. Select a PyTorch/CUDA build compatible "
        f"with this GPU before continuing. Original error: {exc}"
    ) from exc

environment = {
    "captured_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": platform.python_version(),
    "platform": platform.platform(),
    "torch": torch.__version__,
    "torch_cuda": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "transformers": version("transformers"),
    "peft": version("peft"),
    "accelerate": version("accelerate"),
    "bitsandbytes": version("bitsandbytes"),
    "pyyaml": version("PyYAML"),
    "gpu_name": gpu.name,
    "gpu_total_gib": round(gpu_total_gib, 2),
    "gpu_free_gib_at_start": round(gpu_free_gib, 2),
    "gpu_compute_capability": list(compute_capability),
    "torch_compiled_cuda_arches": torch.cuda.get_arch_list(),
    "cuda_probe_checksum": round(cuda_probe_checksum, 6),
    "bf16_supported": bool(torch.cuda.is_bf16_supported()),
    "nvidia_smi": command_output(["nvidia-smi", "--query-gpu=name,driver_version,memory.total,memory.free", "--format=csv,noheader"]),
}

print()
print("=" * 70)
print("  STEP 1 — ENVIRONMENT & GPU CHECK")
print("=" * 70)
print(f"  {'Captured at (UTC)':<30} {environment['captured_at_utc']}")
print(f"  {'Python':<30} {environment['python']}")
print(f"  {'Platform':<30} {environment['platform']}")
print(f"  {'PyTorch':<30} {environment['torch']}")
print(f"  {'CUDA (torch)':<30} {environment['torch_cuda']}")
print(f"  {'cuDNN':<30} {environment['cudnn']}")
print(f"  {'Transformers':<30} {environment['transformers']}")
print(f"  {'PEFT':<30} {environment['peft']}")
print(f"  {'Accelerate':<30} {environment['accelerate']}")
print(f"  {'bitsandbytes':<30} {environment['bitsandbytes']}")
print(f"  {'PyYAML':<30} {environment['pyyaml']}")
print()
print(f"  {'GPU Name':<30} {environment['gpu_name']}")
print(f"  {'GPU Total VRAM (GiB)':<30} {environment['gpu_total_gib']}")
print(f"  {'GPU Free VRAM at start (GiB)':<30} {environment['gpu_free_gib_at_start']}")
print(f"  {'Compute Capability':<30} {environment['gpu_compute_capability']}")
print(f"  {'BF16 Supported':<30} {environment['bf16_supported']}")
print(f"  {'CUDA Probe Checksum':<30} {environment['cuda_probe_checksum']}")
print(f"  {'nvidia-smi':<30} {environment['nvidia_smi']}")
print()
print(f"  CUDA available                 ✅ True")
print(f"  GPU VRAM >= 15 GiB             ✅ {environment['gpu_total_gib']} GiB")
print(f"  Free VRAM >= 12 GiB            ✅ {environment['gpu_free_gib_at_start']} GiB")
print(f"  CUDA kernel probe              ✅ Passed (checksum={environment['cuda_probe_checksum']})")
print("=" * 70)

In [ ]:
# Read the identity and LoRA settings from the versioned M1-B candidate config.
SEED = 298
def find_repo_root(start):
    start = Path(start).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "requirements-models.txt").exists() and (candidate / "models").exists():
            return candidate
    raise FileNotFoundError("Open this notebook from the multiplayer-agent-sessions repository.")

REPO_ROOT = find_repo_root(Path.cwd())
CONFIG_PATH = REPO_ROOT / "eval" / "configs" / "m1-b.yaml"
candidate_config = yaml.safe_load(CONFIG_PATH.read_text(encoding="utf-8"))
CANDIDATE_ID = candidate_config["candidate_id"]
MODEL_ID = candidate_config["model"]["repo"]
MODEL_REVISION = candidate_config["model"]["revision"]
TRAIN_MAX_SEQ_LEN = 1024  # Short smoke run; full value remains in the YAML.
CONTEXT_FEASIBILITY_TOKENS = int(candidate_config["train"]["max_seq_len"])
PHASE1_STEPS = 2
RESUMED_TOTAL_STEPS = 3
LORA_R = int(candidate_config["lora"]["r"])
LORA_ALPHA = int(candidate_config["lora"]["alpha"])
LORA_DROPOUT = float(candidate_config["lora"]["dropout"])
LEARNING_RATE = float(candidate_config["train"]["lr"])
assert CANDIDATE_ID == "m1-b" and candidate_config["mode"] == "lora"
RUN_ID = datetime.now(timezone.utc).strftime("m1-b-qwen3-smoke-%Y%m%dT%H%M%S%fZ")
ARTIFACT_ROOT = Path(os.environ.get("M1_SMOKE_ARTIFACT_ROOT", Path.home() / ".cache" / "multiplayer-agent-sessions" / "m1_gpu_smoke")).expanduser().resolve()
RUN_DIR = ARTIFACT_ROOT / RUN_ID
TRAINER_DIR = RUN_DIR / "trainer"
ADAPTER_DIR = RUN_DIR / "adapter"
RUN_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
compute_dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

print()
print("=" * 70)
print("  STEP 2 — CONFIGURATION & PATHS")
print("=" * 70)
print(f"  {'Repository Root':<30} {REPO_ROOT}")
print(f"  {'Config Path':<30} {CONFIG_PATH}")
print(f"  {'Candidate ID':<30} {CANDIDATE_ID}")
print(f"  {'Model ID':<30} {MODEL_ID}")
print(f"  {'Model Revision':<30} {MODEL_REVISION}")
print(f"  {'Run ID':<30} {RUN_ID}")
print(f"  {'Artifact Root':<30} {ARTIFACT_ROOT}")
print(f"  {'Run Dir':<30} {RUN_DIR}")
print()
print(f"  {'Seed':<30} {SEED}")
print(f"  {'Compute Dtype':<30} {compute_dtype}")
print(f"  {'Train Max Seq Len':<30} {TRAIN_MAX_SEQ_LEN}")
print(f"  {'Context Feasibility Tokens':<30} {CONTEXT_FEASIBILITY_TOKENS}")
print(f"  {'Phase 1 Steps':<30} {PHASE1_STEPS}")
print(f"  {'Resumed Total Steps':<30} {RESUMED_TOTAL_STEPS}")
print(f"  {'LoRA r':<30} {LORA_R}")
print(f"  {'LoRA alpha':<30} {LORA_ALPHA}")
print(f"  {'LoRA dropout':<30} {LORA_DROPOUT}")
print(f"  {'Learning Rate':<30} {LEARNING_RATE}")
print("=" * 70)

In [ ]:
from huggingface_hub import HfApi

resolved_revision = HfApi().model_info(MODEL_ID, revision=MODEL_REVISION).sha
assert resolved_revision == MODEL_REVISION, "The pinned M1-B revision did not resolve exactly."
git_commit = command_output(["git", "-C", str(REPO_ROOT), "rev-parse", "HEAD"])
git_status = command_output(["git", "-C", str(REPO_ROOT), "status", "--short"])

run_manifest = {
    "schema_version": "1.0",
    "run_id": RUN_ID,
    "purpose": "M1-B Qwen3-1.7B QLoRA GPU and checkpoint-resume smoke test",
    "candidate_id": CANDIDATE_ID,
    "is_benchmark_result": False,
    "repository_commit": git_commit,
    "repository_dirty_at_start": bool(git_status),
    "model_id": MODEL_ID,
    "requested_revision": MODEL_REVISION,
    "resolved_revision": resolved_revision,
    "candidate_config": str(CONFIG_PATH.relative_to(REPO_ROOT)),
    "environment": environment,
    "storage": {
        "artifact_root": str(ARTIFACT_ROOT),
        "huggingface_home": os.environ.get("HF_HOME", str(Path.home() / ".cache" / "huggingface")),
    },
    "config": {
        "seed": SEED,
        "train_max_seq_len": TRAIN_MAX_SEQ_LEN,
        "full_config_max_seq_len": int(candidate_config["train"]["max_seq_len"]),
        "context_feasibility_tokens": CONTEXT_FEASIBILITY_TOKENS,
        "phase1_steps": PHASE1_STEPS,
        "resumed_total_steps": RESUMED_TOTAL_STEPS,
        "learning_rate": LEARNING_RATE,
        "lora_r": LORA_R,
        "lora_alpha": LORA_ALPHA,
        "lora_dropout": LORA_DROPOUT,
    },
}

print()
print("=" * 70)
print("  STEP 3 — MODEL REVISION & RUN MANIFEST INIT")
print("=" * 70)
print(f"  {'Model ID':<30} {MODEL_ID}")
print(f"  {'Requested Revision':<30} {MODEL_REVISION}")
print(f"  {'Resolved Revision':<30} {resolved_revision}")
print(f"  {'Revision Match':<30} ✅ Exact match" if resolved_revision == MODEL_REVISION else f"  {'Revision Match':<30} ❌ Mismatch!")
print(f"  {'Repository Commit':<30} {git_commit}")
print(f"  {'Repository Dirty':<30} {'Yes ⚠️' if bool(git_status) else 'No ✅'}")
print(f"  {'Candidate Config':<30} {run_manifest['candidate_config']}")
print(f"  {'Artifact Root':<30} {run_manifest['storage']['artifact_root']}")
print(f"  {'HuggingFace Home':<30} {run_manifest['storage']['huggingface_home']}")
print("=" * 70)

In [ ]:
SYSTEM_PROMPT = (
    "You summarize a multiplayer agent session. Return JSON only with the keys "
    "completed, in_progress, blocked, decisions, and pending_actions. Each item must "
    "contain text and source_event_ids. Do not add facts not supported by the events."
)

smoke_examples = [
    {
        "events": [
            {"event_id": "e1", "actor": "Naman", "text": "The MAST download and checksum script is complete."},
            {"event_id": "e2", "actor": "Pramod", "text": "The evaluation manifest still needs final review."},
        ],
        "target": {
            "completed": [{"text": "MAST download and checksum script completed", "source_event_ids": ["e1"]}],
            "in_progress": [{"text": "Evaluation manifest review", "source_event_ids": ["e2"]}],
            "blocked": [], "decisions": [], "pending_actions": []
        },
    },
    {
        "events": [
            {"event_id": "e3", "actor": "Ayush", "text": "GPU training cannot start until the RTX 5090 reservation is confirmed."},
            {"event_id": "e4", "actor": "Ayush", "text": "I will confirm the reservation by Friday."},
        ],
        "target": {
            "completed": [], "in_progress": [],
            "blocked": [{"text": "GPU training is waiting for an RTX 5090 reservation", "source_event_ids": ["e3"]}],
            "decisions": [],
            "pending_actions": [{"text": "Ayush will confirm the reservation by Friday", "source_event_ids": ["e4"]}]
        },
    },
    {
        "events": [
            {"event_id": "e5", "actor": "Team", "text": "The team selected two-week Scrum cycles."},
            {"event_id": "e6", "actor": "Shriram", "text": "The learning-event schema implementation is in progress."},
        ],
        "target": {
            "completed": [],
            "in_progress": [{"text": "Learning-event schema implementation", "source_event_ids": ["e6"]}],
            "blocked": [],
            "decisions": [{"text": "Use two-week Scrum cycles", "source_event_ids": ["e5"]}],
            "pending_actions": []
        },
    },
]

def user_message(events):
    return "Create the catch-up summary for these session events:\n" + json.dumps(events, ensure_ascii=False)

print()
print("=" * 70)
print("  STEP 4 — SMOKE EXAMPLES & SYSTEM PROMPT")
print("=" * 70)
print(f"  {'Number of smoke examples':<30} {len(smoke_examples)}")
print(f"  {'Expected response keys':<30} completed, in_progress, blocked,")
print(f"  {'':30} decisions, pending_actions")
print()
print("  --- Example 0 (preview) ---")
print(json.dumps(smoke_examples[0], indent=2))
print("=" * 70)

In [ ]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    Trainer,
    TrainingArguments,
    default_data_collator,
)
from transformers.trainer_utils import get_last_checkpoint
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID, revision=resolved_revision, use_fast=True
)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

class CatchupSmokeDataset(Dataset):
    def __init__(self, rows, max_length):
        self.items = []
        for row in rows:
            prompt_messages = [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user_message(row["events"])},
            ]
            full_messages = prompt_messages + [
                {"role": "assistant", "content": json.dumps(row["target"], ensure_ascii=False)}
            ]
            prompt_text = tokenizer.apply_chat_template(
                prompt_messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
            )
            full_text = tokenizer.apply_chat_template(
                full_messages, tokenize=False, add_generation_prompt=False, enable_thinking=False
            )
            prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
            encoded = tokenizer(
                full_text, truncation=True, max_length=max_length, padding="max_length", add_special_tokens=False
            )
            labels = list(encoded["input_ids"])
            prompt_length = min(len(prompt_ids), max_length)
            assert encoded["input_ids"][:prompt_length] == prompt_ids[:prompt_length], (
                "Prompt and full-example tokenization do not share the same prefix."
            )
            labels[:prompt_length] = [-100] * prompt_length
            labels = [label if mask else -100 for label, mask in zip(labels, encoded["attention_mask"])]
            assert any(label != -100 for label in labels), "Target was truncated; raise max_length."
            self.items.append({
                "input_ids": torch.tensor(encoded["input_ids"], dtype=torch.long),
                "attention_mask": torch.tensor(encoded["attention_mask"], dtype=torch.long),
                "labels": torch.tensor(labels, dtype=torch.long),
            })

    def __len__(self):
        return len(self.items)

    def __getitem__(self, index):
        return self.items[index]

train_dataset = CatchupSmokeDataset(smoke_examples, TRAIN_MAX_SEQ_LEN)

print()
print("=" * 70)
print("  STEP 5 — TOKENIZER & DATASET PREPARATION")
print("=" * 70)
print(f"  {'Model ID':<30} {MODEL_ID}")
print(f"  {'Tokenizer Fast':<30} ✅ True")
print(f"  {'Pad Token':<30} {tokenizer.pad_token!r}")
print(f"  {'Padding Side':<30} {tokenizer.padding_side}")
print(f"  {'Vocab Size':<30} {tokenizer.vocab_size}")
print(f"  {'Train Max Seq Len':<30} {TRAIN_MAX_SEQ_LEN}")
print(f"  {'Smoke Examples Prepared':<30} ✅ {len(train_dataset)}")
print("=" * 70)

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
load_started = time.perf_counter()
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    revision=resolved_revision,
    quantization_config=quantization_config,
    device_map={"": 0},
    torch_dtype=compute_dtype,
    attn_implementation="sdpa",
)
model.config.use_cache = False
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

available_suffixes = {name.rsplit(".", 1)[-1] for name, _ in model.named_modules()}
target_modules = list(candidate_config["lora"]["target_modules"])
missing_targets = sorted(set(target_modules) - available_suffixes)
assert not missing_targets, f"Configured LoRA targets are absent from the model: {missing_targets}"

lora_config = LoraConfig(
    task_type="CAUSAL_LM",
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=target_modules,
)
model = get_peft_model(model, lora_config)
load_seconds = time.perf_counter() - load_started
trainable, total = model.get_nb_trainable_parameters()
load_peak_gib = torch.cuda.max_memory_allocated() / 1024**3

run_manifest["model_load"] = {
    "seconds": round(load_seconds, 3),
    "peak_allocated_gib": round(load_peak_gib, 3),
    "target_modules": target_modules,
    "trainable_parameters": trainable,
    "total_parameters": total,
    "trainable_percent": round(100 * trainable / total, 6),
}

print()
print("=" * 70)
print("  STEP 6 — MODEL LOAD & LORA SETUP")
print("=" * 70)
print(f"  {'Model':<30} ✅ {MODEL_ID}")
print(f"  {'Quantization':<30} 4-bit NF4 (double quant)")
print(f"  {'Compute Dtype':<30} {compute_dtype}")
print(f"  {'Attention Implementation':<30} sdpa")
print(f"  {'LoRA r':<30} {LORA_R}")
print(f"  {'LoRA alpha':<30} {LORA_ALPHA}")
print(f"  {'LoRA dropout':<30} {LORA_DROPOUT}")
print(f"  {'LoRA Target Modules':<30} {target_modules}")
print()
print(f"  --- Load Summary ---")
print(f"  {'Load Time (s)':<30} {run_manifest['model_load']['seconds']}")
print(f"  {'Peak VRAM (GiB)':<30} {run_manifest['model_load']['peak_allocated_gib']}")
print(f"  {'Trainable Parameters':<30} {trainable:,}")
print(f"  {'Total Parameters':<30} {total:,}")
print(f"  {'Trainable %':<30} {run_manifest['model_load']['trainable_percent']}%")
print("=" * 70)

In [ ]:
def training_arguments(max_steps):
    return TrainingArguments(
        output_dir=str(TRAINER_DIR),
        overwrite_output_dir=False,
        max_steps=max_steps,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        learning_rate=LEARNING_RATE,
        lr_scheduler_type="constant",
        warmup_steps=0,
        logging_steps=1,
        save_strategy="steps",
        save_steps=1,
        save_total_limit=3,
        save_safetensors=True,
        optim="paged_adamw_8bit",
        bf16=torch.cuda.is_bf16_supported(),
        fp16=not torch.cuda.is_bf16_supported(),
        gradient_checkpointing=True,
        remove_unused_columns=False,
        dataloader_num_workers=0,
        report_to=[],
        seed=SEED,
        data_seed=SEED,
    )

torch.cuda.reset_peak_memory_stats()
phase1_started = time.perf_counter()
trainer = Trainer(
    model=model,
    args=training_arguments(PHASE1_STEPS),
    train_dataset=train_dataset,
    data_collator=default_data_collator,
)
phase1_result = trainer.train()
phase1_seconds = time.perf_counter() - phase1_started
phase1_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
checkpoint = get_last_checkpoint(str(TRAINER_DIR))
assert math.isfinite(float(phase1_result.training_loss)), "Phase 1 produced a non-finite loss."
assert checkpoint is not None, "Trainer did not create a checkpoint."
checkpoint_path = Path(checkpoint)
required_checkpoint_files = {
    "adapter_config.json", "adapter_model.safetensors", "optimizer.pt",
    "scheduler.pt", "trainer_state.json",
}
missing_checkpoint_files = sorted(
    name for name in required_checkpoint_files if not (checkpoint_path / name).is_file()
)
assert not missing_checkpoint_files, f"Checkpoint is incomplete: {missing_checkpoint_files}"

run_manifest["phase1_training"] = {
    "seconds": round(phase1_seconds, 3),
    "peak_allocated_gib": round(phase1_peak_gib, 3),
    "checkpoint": str(checkpoint_path),
    "checkpoint_files": sorted(path.name for path in checkpoint_path.iterdir()),
    "metrics": phase1_result.metrics,
}

print()
print("=" * 70)
print("  STEP 7 — PHASE 1 TRAINING (BACKWARD SMOKE TEST)")
print("=" * 70)
print(f"  {'Steps Run':<30} {PHASE1_STEPS}")
print(f"  {'Training Loss':<30} {phase1_result.training_loss}")
print(f"  {'Loss Finite':<30} ✅ Yes" if math.isfinite(float(phase1_result.training_loss)) else f"  {'Loss Finite':<30} ❌ Non-finite!")
print(f"  {'Training Time (s)':<30} {run_manifest['phase1_training']['seconds']}")
print(f"  {'Peak VRAM (GiB)':<30} {run_manifest['phase1_training']['peak_allocated_gib']}")
print()
print(f"  --- Checkpoint Info ---")
print(f"  {'Checkpoint Path':<30} {checkpoint_path}")
print(f"  {'Checkpoint Created':<30} ✅ Yes")
print(f"  {'Files in Checkpoint':<30} {run_manifest['phase1_training']['checkpoint_files']}")
print(f"  {'Missing Files':<30} {'None ✅' if not missing_checkpoint_files else str(missing_checkpoint_files) + ' ❌'}")
print("=" * 70)

In [ ]:
# Recreate the Trainer and continue beyond the saved global step. A constant
# learning-rate schedule is used so changing max_steps does not alter a decay curve.
del trainer
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
resume_started = time.perf_counter()
resumed_trainer = Trainer(
    model=model,
    args=training_arguments(RESUMED_TOTAL_STEPS),
    train_dataset=train_dataset,
    data_collator=default_data_collator,
)
resume_result = resumed_trainer.train(resume_from_checkpoint=str(checkpoint))
resume_seconds = time.perf_counter() - resume_started
resume_peak_gib = torch.cuda.max_memory_allocated() / 1024**3
assert resumed_trainer.state.global_step == RESUMED_TOTAL_STEPS, (
    f"Expected global step {RESUMED_TOTAL_STEPS}; got {resumed_trainer.state.global_step}."
)
assert math.isfinite(float(resume_result.training_loss)), "Resume phase produced a non-finite loss."

resumed_trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
assert (ADAPTER_DIR / "adapter_config.json").is_file()
assert (ADAPTER_DIR / "adapter_model.safetensors").is_file()
run_manifest["resume_training"] = {
    "resumed_from": str(checkpoint),
    "final_global_step": resumed_trainer.state.global_step,
    "seconds": round(resume_seconds, 3),
    "peak_allocated_gib": round(resume_peak_gib, 3),
    "adapter_dir": str(ADAPTER_DIR),
    "metrics": resume_result.metrics,
}

print()
print("=" * 70)
print("  STEP 8 — CHECKPOINT RESUME & ADAPTER SAVE")
print("=" * 70)
print(f"  {'Resumed From':<30} {checkpoint}")
print(f"  {'Final Global Step':<30} {resumed_trainer.state.global_step}")
print(f"  {'Expected Total Steps':<30} {RESUMED_TOTAL_STEPS}")
print(f"  {'Resume Succeeded':<30} ✅ Yes" if resumed_trainer.state.global_step == RESUMED_TOTAL_STEPS else f"  {'Resume Succeeded':<30} ❌ Step mismatch!")
print(f"  {'Resume Loss Finite':<30} ✅ Yes" if math.isfinite(float(resume_result.training_loss)) else f"  {'Resume Loss Finite':<30} ❌ Non-finite!")
print(f"  {'Training Time (s)':<30} {run_manifest['resume_training']['seconds']}")
print(f"  {'Peak VRAM (GiB)':<30} {run_manifest['resume_training']['peak_allocated_gib']}")
print()
print(f"  --- Adapter Save Info ---")
print(f"  {'Adapter Dir':<30} {ADAPTER_DIR}")
print(f"  {'adapter_config.json':<30} ✅ Present" if (ADAPTER_DIR / "adapter_config.json").is_file() else f"  {'adapter_config.json':<30} ❌ Missing!")
print(f"  {'adapter_model.safetensors':<30} ✅ Present" if (ADAPTER_DIR / "adapter_model.safetensors").is_file() else f"  {'adapter_model.safetensors':<30} ❌ Missing!")
print("=" * 70)

In [ ]:
inference_events = [
    {"event_id": "demo-1", "actor": "Naman", "text": "The MAST manifest validation tests passed."},
    {"event_id": "demo-2", "actor": "Ayush", "text": "The M1-B Qwen3 smoke run is in progress on the RTX 4090."},
    {"event_id": "demo-3", "actor": "Team", "text": "The team decided that lessons require human approval."},
]
messages = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": user_message(inference_events)},
]
inputs = tokenizer.apply_chat_template(
    messages, tokenize=True, add_generation_prompt=True, enable_thinking=False,
    return_tensors="pt", return_dict=True
).to(model.device)

model.gradient_checkpointing_disable()
model.config.use_cache = True
model.eval()
torch.cuda.reset_peak_memory_stats()
torch.cuda.synchronize()
inference_started = time.perf_counter()
with torch.inference_mode():
    generated = model.generate(
        **inputs,
        max_new_tokens=192,
        do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        eos_token_id=tokenizer.eos_token_id,
    )
torch.cuda.synchronize()
inference_seconds = time.perf_counter() - inference_started
new_tokens = generated[0, inputs["input_ids"].shape[1]:]
response_text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

def parse_json_object(text):
    start, end = text.find("{"), text.rfind("}")
    if start < 0 or end < start:
        return None
    try:
        return json.loads(text[start:end + 1])
    except json.JSONDecodeError:
        return None

parsed_response = parse_json_object(response_text)
expected_response_keys = {"completed", "in_progress", "blocked", "decisions", "pending_actions"}
allowed_source_event_ids = {event["event_id"] for event in inference_events}
response_contract_valid = (
    isinstance(parsed_response, dict)
    and set(parsed_response) == expected_response_keys
    and all(isinstance(parsed_response[key], list) for key in expected_response_keys)
    and all(
        isinstance(item, dict)
        and isinstance(item.get("text"), str)
        and isinstance(item.get("source_event_ids"), list)
        and all(
            isinstance(event_id, str) and event_id in allowed_source_event_ids
            for event_id in item["source_event_ids"]
        )
        for key in expected_response_keys for item in parsed_response[key]
    )
)
run_manifest["inference"] = {
    "seconds": round(inference_seconds, 3),
    "generated_tokens": int(new_tokens.numel()),
    "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    "valid_json_object": parsed_response is not None,
    "response_contract_valid": response_contract_valid,
    "response": response_text,
}

print()
print("=" * 70)
print("  STEP 9 — INFERENCE (M1 CATCH-UP RESPONSE GENERATION)")
print("=" * 70)
print(f"  {'Input Events':<30} {len(inference_events)}")
print(f"  {'Max New Tokens':<30} 192")
print(f"  {'Generated Tokens':<30} {run_manifest['inference']['generated_tokens']}")
print(f"  {'Inference Time (s)':<30} {run_manifest['inference']['seconds']}")
print(f"  {'Peak VRAM (GiB)':<30} {run_manifest['inference']['peak_allocated_gib']}")
print()
print(f"  {'Valid JSON Object':<30} {'✅ Yes' if parsed_response is not None else '❌ No'}")
print(f"  {'Response Contract Valid':<30} {'✅ Yes' if response_contract_valid else '❌ No'}")
print()
print("  --- Raw Model Response ---")
print(response_text)
print("=" * 70)

In [ ]:
# Test the proposed 4096-token context separately. This is a forward-pass
# feasibility check, not a full 4096-token training benchmark.
event_text = "event_id=ctx actor=team status=in_progress evidence=source_event "
base_ids = tokenizer(event_text, add_special_tokens=False)["input_ids"]
repeats = math.ceil(CONTEXT_FEASIBILITY_TOKENS / len(base_ids))
context_ids = (base_ids * repeats)[:CONTEXT_FEASIBILITY_TOKENS]
context_tensor = torch.tensor([context_ids], dtype=torch.long, device=model.device)
context_mask = torch.ones_like(context_tensor)

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
context_started = time.perf_counter()
context_error = None
try:
    with torch.inference_mode():
        context_output = model(input_ids=context_tensor, attention_mask=context_mask, use_cache=False)
    torch.cuda.synchronize()
    context_passed = tuple(context_output.logits.shape[:2]) == (1, CONTEXT_FEASIBILITY_TOKENS)
    del context_output
except torch.cuda.OutOfMemoryError as exc:
    context_passed = False
    context_error = f"CUDA out of memory: {exc}"
    torch.cuda.empty_cache()
except RuntimeError as exc:
    context_passed = False
    context_error = f"Runtime error during 4096-token check: {exc}"
    torch.cuda.empty_cache()

run_manifest["context_feasibility"] = {
    "tokens": CONTEXT_FEASIBILITY_TOKENS,
    "passed": context_passed,
    "seconds": round(time.perf_counter() - context_started, 3),
    "peak_allocated_gib": round(torch.cuda.max_memory_allocated() / 1024**3, 3),
    "error": context_error,
}

print()
print("=" * 70)
print("  STEP 10 — 4096-TOKEN CONTEXT FEASIBILITY CHECK")
print("=" * 70)
print(f"  {'Context Tokens':<30} {CONTEXT_FEASIBILITY_TOKENS}")
print(f"  {'Forward Pass':<30} {'✅ PASSED' if context_passed else '❌ FAILED'}")
print(f"  {'Time (s)':<30} {run_manifest['context_feasibility']['seconds']}")
print(f"  {'Peak VRAM (GiB)':<30} {run_manifest['context_feasibility']['peak_allocated_gib']}")
if context_error:
    print(f"  {'Error':<30} ⚠️  {context_error}")
else:
    print(f"  {'Error':<30} None")
print("=" * 70)

In [ ]:
run_manifest["completed_at_utc"] = datetime.now(timezone.utc).isoformat()
run_manifest["acceptance"] = {
    "cuda_available": True,
    "model_loaded_4bit": True,
    "backward_steps_completed": int(resumed_trainer.state.global_step),
    "checkpoint_created": checkpoint_path.is_dir() and not missing_checkpoint_files,
    "checkpoint_resumed": resumed_trainer.state.global_step >= RESUMED_TOTAL_STEPS,
    "adapter_saved": ADAPTER_DIR.exists(),
    "inference_completed": bool(response_text),
    "response_contract_valid": response_contract_valid,
    "context_4096_forward_passed": context_passed,
}
required_infrastructure_checks = [
    "cuda_available", "model_loaded_4bit", "checkpoint_created",
    "checkpoint_resumed", "adapter_saved", "inference_completed",
    "context_4096_forward_passed",
]
run_manifest["smoke_passed"] = all(
    bool(run_manifest["acceptance"][name]) for name in required_infrastructure_checks
)
manifest_path = RUN_DIR / "run_manifest.json"
manifest_path.write_text(json.dumps(run_manifest, indent=2, default=str) + "\n", encoding="utf-8")

acc = run_manifest["acceptance"]
overall = run_manifest["smoke_passed"]

print()
print("=" * 70)
print("  FINAL ACCEPTANCE REPORT — M1-B Qwen3-1.7B GPU Smoke Test")
print("=" * 70)
print(f"  {'cuda_available':<35} {'✅' if acc['cuda_available'] else '❌'} {acc['cuda_available']}")
print(f"  {'model_loaded_4bit':<35} {'✅' if acc['model_loaded_4bit'] else '❌'} {acc['model_loaded_4bit']}")
print(f"  {'backward_steps_completed':<35} {'✅' if acc['backward_steps_completed'] >= RESUMED_TOTAL_STEPS else '❌'} {acc['backward_steps_completed']}")
print(f"  {'checkpoint_created':<35} {'✅' if acc['checkpoint_created'] else '❌'} {acc['checkpoint_created']}")
print(f"  {'checkpoint_resumed':<35} {'✅' if acc['checkpoint_resumed'] else '❌'} {acc['checkpoint_resumed']}")
print(f"  {'adapter_saved':<35} {'✅' if acc['adapter_saved'] else '❌'} {acc['adapter_saved']}")
print(f"  {'inference_completed':<35} {'✅' if acc['inference_completed'] else '❌'} {acc['inference_completed']}")
print(f"  {'response_contract_valid':<35} {'✅' if acc['response_contract_valid'] else '⚠️ '} {acc['response_contract_valid']}")
print(f"  {'context_4096_forward_passed':<35} {'✅' if acc['context_4096_forward_passed'] else '❌'} {acc['context_4096_forward_passed']}")
print("-" * 70)
print(f"  {'OVERALL':<35} {'✅ SMOKE TEST PASSED' if overall else '❌ SMOKE TEST FAILED'}")
print("=" * 70)
print()
print(f"  Run manifest written to: {manifest_path}")
print()
print("  --- Machine-Readable Manifest (acceptance section) ---")
print(json.dumps(run_manifest["acceptance"], indent=2))
assert run_manifest["smoke_passed"], f"One or more infrastructure checks failed; inspect {manifest_path}."

## Evidence to attach to M1-04

After the notebook finishes, Ayush should attach or link the following without committing model weights:

- the completed notebook with outputs;
- `run_manifest.json`;
- the small adapter configuration and tokenizer metadata, if repository policy permits;
- the checkpoint path and storage location;
- the final acceptance summary;
- a note that the run used an RTX 5080 Laptop, 16 GB, and `eval/configs/m1-b.yaml`.

Do not report the smoke-test loss or generated response as an M1 benchmark result. Full M1 evaluation must use the frozen dataset, split, metrics and candidate contract from M1-01 and M1-02.